# Week 2 — Data, APIs & Embeddings
## Practical AI/ML notebook: from raw data to semantic search

**Course:** AI/ML AWS AASTU Cohort  
**Goal:** Practice the full pipeline: inspect data, clean it, call an API, prepare text, create embeddings, compare vectors, and build a small semantic-search system.

### How to use this notebook
- Run cells from top to bottom.
- Read each short explanation before running its code.
- Change the examples and queries to see how the results change.
- This notebook uses a small, self-contained FAQ dataset, so the core lab works without downloading data or needing an API key.
- The optional API section calls a public demo API. Network access may be unavailable in some classroom environments.

**Deliverable:** Save this notebook, add your own FAQ rows and example queries, and write a short README explaining how to run it.

## 1. Set up the environment

**Why these libraries?**
- **NumPy**: fast numerical arrays and vector operations.
- **Pandas**: tabular data, cleaning, CSV and JSON.
- **Requests**: send HTTP requests to APIs.
- **Matplotlib**: make a simple chart during exploration.
- **Sentence Transformers**: turn text into meaningful numeric vectors.
- **scikit-learn**: calculate cosine similarity.
- **ChromaDB**: store text and search it by vector similarity.

Run this cell once in your notebook environment. If a package installation fails, ask your instructor before changing the environment.

In [ ]:
%pip install -q numpy pandas matplotlib requests scikit-learn sentence-transformers chromadb

## 2. Import the tools

Imports make library functions available in this notebook. We will use them in later cells.

In [ ]:
import json
import re
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.metrics.pairwise import cosine_similarity

print("Imports completed.")

## 3. NumPy foundations: arrays and vectorization

A **NumPy array** stores numerical values in a structure that supports fast calculations.  
**Vectorization** means applying an operation to many values at once rather than writing a Python loop.

We use it because numerical work becomes shorter and often faster.

In [ ]:
prices = np.array([85, 120, 200, 150])

print("Prices:", prices)
print("Shape:", prices.shape)       # Number of elements in this one-dimensional array
print("Mean:", prices.mean())       # Average price
print("Maximum:", prices.max())     # Highest price
print("Prices with 15% added:", prices * 1.15)

### Try it
Change the prices in the array. Notice how the shape and summary values change.

In [ ]:
# A two-dimensional array has rows and columns.
measurements = np.array([[10, 20, 30], [15, 25, 35]])

print(measurements)
print("Rows, columns:", measurements.shape)
print("Column averages:", measurements.mean(axis=0))

## 4. Pandas foundations: DataFrames and Series

A **DataFrame** is a table with rows and named columns. A **Series** is one column. Pandas is useful when data has both text and numbers and we need to filter, sort, summarize, or clean it.

In [ ]:
menu = pd.DataFrame({
    "item": ["Shiro", "Firfir", "Tibs", "Beyaynet"],
    "price": [85, 120, 200, 150],
    "city": ["Addis Ababa", "Addis Ababa", "Adama", "Addis Ababa"]
})

display(menu)
print("Price column:")
display(menu["price"])
print("Items below 150:")
display(menu[menu["price"] < 150])
print("Average price:", menu["price"].mean())
display(menu.sort_values("price"))

## 5. Create a realistic, intentionally messy dataset

Real datasets often contain missing values, duplicates, inconsistent spelling, extra spaces, and numbers stored as text. We will create a small dataset with these problems so that the cleaning steps are visible.

In [ ]:
raw_orders = pd.DataFrame({
    "order_id": [1, 2, 2, 3, 4],
    "city": [" addis ababa ", "Addis Ababa", "Addis Ababa", "Adama", "Hawassa"],
    "price": ["1,200", None, None, "950", "1,500"],
    "review": [
        "Fast delivery and helpful support.",
        "My package arrived late.",
        "My package arrived late.",
        "Affordable and reliable service.",
        "I need help tracking my order."
    ]
})

display(raw_orders)
print("Data types before cleaning:")
print(raw_orders.dtypes)

## 6. Clean the data

Cleaning helps prevent incorrect summaries and unreliable model inputs.

- **Duplicates:** remove repeated records when they are genuinely duplicates.
- **Missing values:** fill them or remove the affected rows based on the problem.
- **Types:** convert text-formatted numbers to numeric values.
- **Normalization:** make comparable numeric features use a consistent scale. This is not the same as text normalization.

Here, we remove exact duplicate rows, standardize city names, convert prices, and fill missing prices with the median. In a real project, decide whether duplicate order IDs represent repeated records or legitimate separate events before dropping them.

In [ ]:
clean = raw_orders.drop_duplicates().copy()

# Remove surrounding spaces and standardize city names.
clean["city"] = clean["city"].str.strip().str.title()

# Remove commas, then convert the price strings to numbers.
clean["price"] = pd.to_numeric(
    clean["price"].str.replace(",", "", regex=False),
    errors="coerce"
)

# Fill missing prices with the median of the available prices.
median_price = clean["price"].median()
clean["price"] = clean["price"].fillna(median_price)

print("Median used for missing prices:", median_price)
display(clean)
print("Missing values per column:")
print(clean.isna().sum())
print("Types after cleaning:")
print(clean.dtypes)

### Numeric normalization: rescale values to 0–1

Min-max normalization maps the minimum value to 0 and maximum to 1. It can help when features have very different scales. Do not apply it automatically to every column; it depends on the task and model.

In [ ]:
minimum = clean["price"].min()
maximum = clean["price"].max()

if maximum != minimum:
    clean["price_0_to_1"] = (clean["price"] - minimum) / (maximum - minimum)
else:
    clean["price_0_to_1"] = 0.0

display(clean[["price", "price_0_to_1"]])

## 7. Explore a dataset before modeling

Always inspect a dataset before using it. These commands quickly reveal its size, sample rows, data types, summary statistics, and missing values.

In [ ]:
print("Rows and columns:", clean.shape)
display(clean.head())
clean.info()
display(clean.describe(include="all"))
print("Missing values:")
print(clean.isna().sum())

### Visualize one column

A plot can make distributions easier to understand than a table alone. This chart shows the cleaned order prices.

In [ ]:
clean["price"].plot(kind="bar", title="Cleaned order prices", xlabel="Row", ylabel="Price")
plt.tight_layout()
plt.show()

## 8. Save and load CSV and JSON

CSV is convenient for flat tables. JSON is common for API responses and nested data. Use the format that fits the source and structure.

In [ ]:
# Save a table as CSV, then read it back.
clean.to_csv("clean_orders.csv", index=False)
csv_loaded = pd.read_csv("clean_orders.csv")
display(csv_loaded.head())

# JSON can represent records as a list of objects.
courses = [
    {"code": "CS101", "title": "Introduction to AI", "credits": 3},
    {"code": "DS201", "title": "Data Foundations", "credits": 4}
]

with open("courses.json", "w", encoding="utf-8") as file:
    json.dump(courses, file, indent=2)

with open("courses.json", "r", encoding="utf-8") as file:
    courses_loaded = json.load(file)

print(courses_loaded)
display(pd.DataFrame(courses_loaded))

## 9. APIs and JSON: make a request

An **API** lets one program request data or a service from another. An HTTP request has a method, URL, and sometimes parameters or headers. The response includes a status code and often JSON data.

Common status codes:
- `200`: request succeeded.
- `401`: authentication is missing or invalid.
- `404`: resource not found.
- `429`: rate limit reached.

This example uses a public demo endpoint. If your network blocks it, skip the cell and continue with the local JSON example.

In [ ]:
api_url = "https://jsonplaceholder.typicode.com/todos/1"

try:
    response = requests.get(api_url, timeout=10)
    print("Status code:", response.status_code)
    response.raise_for_status()  # Raise an error for unsuccessful HTTP status codes.
    api_data = response.json()   # Parse the JSON response into Python objects.
    print(api_data)
except requests.RequestException as error:
    print("API request did not work in this environment:", error)

### API authentication basics

Some APIs require a token. Never commit a real key to GitHub or place it directly in a shared notebook. Read it from an environment variable and follow the API provider's instructions for the correct header and permissions.

This is a template, not a working endpoint:

In [ ]:
# Example template only — set API_KEY in your environment before using a real API.
# api_key = os.environ.get("API_KEY")
# if not api_key:
#     raise ValueError("Set the API_KEY environment variable first.")
#
# response = requests.get(
#     "https://api.example.com/resource",
#     headers={"Authorization": f"Bearer {api_key}"},
#     timeout=10
# )
# response.raise_for_status()
# data = response.json()

## 10. Text preparation: clean and tokenize

Text data is often inconsistent. **Text cleaning** standardizes it; **tokenization** splits text into smaller units called tokens. This simple example splits on spaces after removing punctuation. Real language models often use more sophisticated subword tokenizers, so this is an introduction rather than a production tokenizer.

In [ ]:
sample_text = "  Visit AASTU!! Classes START Monday.  "

def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

cleaned_text = clean_text(sample_text)
tokens = cleaned_text.split()

print("Original:", repr(sample_text))
print("Cleaned:", cleaned_text)
print("Tokens:", tokens)

## 11. Chunking: split long documents into manageable pieces

An embedding model has a context limit, and searching a whole long handbook may return too much unrelated information. **Chunking** divides text into smaller overlapping pieces. Overlap helps preserve context at chunk boundaries.

Chunk size is a design choice: tiny chunks can lose context; huge chunks can mix unrelated ideas. We will use word-based chunks for a simple demonstration.

In [ ]:
def chunk_text(text, chunk_size=35, overlap=8):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    words = text.split()
    step = chunk_size - overlap
    return [
        " ".join(words[start:start + chunk_size])
        for start in range(0, len(words), step)
    ]

long_text = (
    "Students should register for courses through the student portal. "
    "Registration dates are announced before each semester. "
    "If the portal is unavailable, contact the registrar office. "
    "Keep your student ID ready when asking for support. "
) * 4

chunks_preview = chunk_text(long_text, chunk_size=25, overlap=5)
for number, chunk in enumerate(chunks_preview, start=1):
    print(f"Chunk {number}: {chunk}\n")

## 12. Embeddings: represent text as vectors

An **embedding** is a list of numbers produced by a model to represent a piece of text. The model learns representations from language examples so that texts with related meanings often have vectors that are close in the embedding space.

Why use them? Keyword search can miss related wording. For example, a user might ask “I can't access my account” while a document says “reset your password.” Embeddings can help match these phrases even when they do not share all the same words.

The next cell loads a small sentence-transformer model. The first run may download model files and may take a while. If model download is blocked, the later semantic-search section will not run until the model is available.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

example_sentences = [
    "I forgot my password.",
    "I cannot log in to my account.",
    "How do I make injera?"
]

example_vectors = model.encode(example_sentences, normalize_embeddings=True)

print("Number of sentences:", len(example_sentences))
print("Vector matrix shape:", example_vectors.shape)
print("First vector preview:", example_vectors[0][:10])

## 13. Cosine similarity: compare vector direction

Cosine similarity compares the direction of two vectors. With normalized embeddings, the dot product gives cosine similarity directly. Higher scores mean the model considers the texts more similar; the score is not a probability or a guarantee that a result is correct.

In [ ]:
similarity_matrix = cosine_similarity(example_vectors)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=example_sentences,
    columns=example_sentences
)
display(similarity_df.round(3))

### Nearest neighbors: retrieve the top-k matches

For a query, score every document, sort by score, and return the top `k`. This is called nearest-neighbor retrieval. The simple method below checks every item, which is fine for a small dataset but becomes expensive for very large collections.

In [ ]:
query = "I can't access my account"
query_vector = model.encode([query], normalize_embeddings=True)

scores = cosine_similarity(query_vector, example_vectors)[0]
top_k = 2
best_indices = np.argsort(scores)[::-1][:top_k]

for index in best_indices:
    print(f"Score: {scores[index]:.3f} | {example_sentences[index]}")

## 14. Build the semantic-search FAQ dataset

Now we build the lab deliverable. Each row has an ID, a question, an answer, and a category. We embed both the question and answer together so retrieval can match either the way a user asks or the information in the answer.

Add, edit, or replace these example FAQs with your own domain dataset.

In [ ]:
faqs = pd.DataFrame([
    {
        "id": "faq_001",
        "question": "How do I reset my password?",
        "answer": "Open the sign-in page and choose Forgot Password. Follow the instructions sent to your registered email.",
        "category": "account"
    },
    {
        "id": "faq_002",
        "question": "I cannot log in to my account.",
        "answer": "Check that your email and password are correct. If you still cannot sign in, reset your password.",
        "category": "account"
    },
    {
        "id": "faq_003",
        "question": "When does course registration open?",
        "answer": "Course registration dates are announced before the start of each semester in the student portal.",
        "category": "registration"
    },
    {
        "id": "faq_004",
        "question": "How can I contact the registrar?",
        "answer": "Contact the registrar office through the official student support channel during working hours.",
        "category": "support"
    },
    {
        "id": "faq_005",
        "question": "Where can I see my grades?",
        "answer": "Sign in to the student portal and open the academic results or grades section.",
        "category": "academics"
    },
    {
        "id": "faq_006",
        "question": "How do I pay my tuition?",
        "answer": "Follow the payment instructions published by the institution and keep your payment receipt.",
        "category": "finance"
    },
    {
        "id": "faq_007",
        "question": "I need help accessing the online learning platform.",
        "answer": "Verify your account details and internet connection. Contact support if the platform remains unavailable.",
        "category": "support"
    },
    {
        "id": "faq_008",
        "question": "How do I download my academic transcript?",
        "answer": "Check the student portal for transcript services or ask the registrar office about the official process.",
        "category": "academics"
    }
])

display(faqs)

### Prepare and embed the FAQ text

Each FAQ becomes one searchable document. We retain the original fields as metadata so that the result can show the question, answer, and category rather than only a vector score.

In [ ]:
faqs["search_text"] = (
    "Question: " + faqs["question"] + "\nAnswer: " + faqs["answer"]
)

faq_vectors = model.encode(
    faqs["search_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("FAQ count:", len(faqs))
print("Embedding matrix shape:", faq_vectors.shape)

## 15. Write a simple semantic-search function

This first version uses NumPy and cosine similarity directly. It makes the retrieval logic easy to inspect. Later, we will put the same documents into ChromaDB, a vector database.

In [ ]:
def semantic_search(query, documents=faqs, vectors=faq_vectors, top_k=3):
    query_vector = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_vector, vectors)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = documents.iloc[top_indices][
        ["id", "question", "answer", "category"]
    ].copy()
    results["similarity"] = scores[top_indices]
    return results.reset_index(drop=True)

display(semantic_search("I forgot my login details", top_k=3))

### Test multiple queries

A semantic-search system should be tested with different phrasings, not just one query. Check whether the top results are genuinely useful. Similarity scores are ranking signals, not a substitute for evaluating answer quality.

In [ ]:
test_queries = [
    "I cannot remember my password",
    "When can I sign up for classes?",
    "Where are my exam results?",
    "I need a copy of my transcript",
    "How can I pay the school fees?"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)
    results = semantic_search(query, top_k=2)
    for _, row in results.iterrows():
        print(f"({row['similarity']:.3f}) {row['question']}")
        print(row["answer"])

## 16. Add a simple category filter

Real search often combines semantic similarity with structured filters. For example, search for the meaning of a query but restrict results to `academics` or `support`. We can filter before scoring.

In [ ]:
def filtered_semantic_search(query, category=None, top_k=3):
    candidates = faqs if category is None else faqs[faqs["category"] == category]

    if candidates.empty:
        return pd.DataFrame(columns=["id", "question", "answer", "category", "similarity"])

    candidate_vectors = model.encode(
        candidates["search_text"].tolist(),
        normalize_embeddings=True
    )
    query_vector = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_vector, candidate_vectors)[0]
    indices = np.argsort(scores)[::-1][:top_k]

    results = candidates.iloc[indices][
        ["id", "question", "answer", "category"]
    ].copy()
    results["similarity"] = scores[indices]
    return results.reset_index(drop=True)

display(filtered_semantic_search("Where can I get my transcript?", category="academics"))

## 17. Vector databases: why they exist

A vector database stores embeddings and associated information and provides retrieval operations. For a small dataset, comparing a query against every vector is simple. At large scale, vector databases can use indexes to find likely neighbors faster.

- **Brute force:** compare the query with every stored vector; straightforward, but work grows with the dataset.
- **Approximate nearest-neighbor (ANN) indexes:** trade some exactness for faster retrieval.
- **Metadata:** lets us retain IDs, categories, and original text alongside vectors.

We will use ChromaDB in local persistent mode. It stores data on disk in the `chroma_faq_db` directory.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_faq_db")

# Reuse the collection if this cell is run more than once.
collection_name = "aastu_week2_faqs"
existing_names = [collection.name for collection in chroma_client.list_collections()]

if collection_name in existing_names:
    chroma_client.delete_collection(collection_name)

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "Week 2 semantic search FAQ demo"}
)

collection.add(
    ids=faqs["id"].tolist(),
    documents=faqs["search_text"].tolist(),
    metadatas=[
        {
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"]
        }
        for _, row in faqs.iterrows()
    ],
    embeddings=faq_vectors.tolist()
)

print("Stored FAQ documents:", collection.count())

## 18. Query ChromaDB

Chroma can store the embeddings we generated and return the closest documents. Because we supplied embeddings when adding documents, we also supply a query embedding when searching. This keeps the model used for embedding explicit.

In [ ]:
def chroma_search(query, top_k=3, category=None):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    where_filter = {"category": category} if category else None

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    output = []
    for i, doc_id in enumerate(results["ids"][0]):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]
        output.append({
            "id": doc_id,
            "question": metadata["question"],
            "answer": metadata["answer"],
            "category": metadata["category"],
            "distance": distance
        })

    return pd.DataFrame(output)

display(chroma_search("I am locked out and cannot sign in", top_k=3))

### Compare the two approaches

The direct NumPy approach is transparent and useful for learning. ChromaDB adds storage and database-style retrieval, which is more convenient as the dataset grows. Their ranking may differ slightly because of distance conventions, embedding configuration, or index behavior.

In [ ]:
query = "When can I register for my classes?"

print("Direct similarity search:")
display(semantic_search(query, top_k=3))

print("ChromaDB search:")
display(chroma_search(query, top_k=3))

## 19. Make the notebook interactive

This small helper lets a student enter a query and see the top results. It is a notebook demo, not a deployed application.

In [ ]:
def ask_faq():
    query = input("Ask a question: ").strip()
    if not query:
        print("Please enter a question.")
        return

    results = chroma_search(query, top_k=3)
    for rank, row in results.iterrows():
        print(f"\nResult {rank + 1}: {row['question']}")
        print(f"Category: {row['category']}")
        print(f"Answer: {row['answer']}")
        print(f"Distance: {row['distance']:.4f}")

# Run this function when you want to ask your own question:
# ask_faq()

## 20. Lab exercises

Complete these before submitting:

1. Add at least five FAQs from a topic you know well.
2. Try five differently worded queries and inspect the top three results.
3. Add a new category and test the category filter.
4. Deliberately add a duplicate row and a missing value, then demonstrate how you detect and clean them.
5. Compare direct cosine-similarity results with ChromaDB results.
6. Write down one query where search performs poorly. Explain whether the issue may come from the dataset, chunk size, embedding model, or query wording.

### Optional extension
Try chunking a longer policy or handbook into paragraphs, embedding each chunk, and returning the best matching passages. Preserve a source title and chunk number as metadata so users can trace each result to its origin.

## 21. Troubleshooting notes

- **Package installation fails:** check the Python environment and internet access; restart the kernel after installing if imports fail.
- **Model download fails:** `SentenceTransformer` needs the model files available locally or downloadable. Use a network-enabled environment or ask the instructor for an approved pre-downloaded model.
- **API request fails:** the network may block the endpoint. The rest of the notebook can still run using local data.
- **Chroma collection errors:** rerun the setup cell, which deletes and recreates this demo collection. Do not use that reset pattern for a real production database.
- **Poor search results:** inspect the source text, add better examples, change chunk boundaries, and evaluate several realistic queries.

## Key takeaway

A practical semantic-search pipeline is:

**Load data → clean it → prepare text → chunk it if needed → embed it → store vectors and metadata → embed the query → retrieve nearest matches → evaluate the results.**

For production systems, also consider privacy, access control, monitoring, model/version consistency, evaluation datasets, and the cost of embedding and storage.